In [1]:
import numpy as np  # linear algebra
import pandas as pd  # data processing, CSV file I/O (e.g. pd.read_csv)
import os
import copy
import glob
import random
from random import random as rd
import gc



In [2]:
TRAIN_PATH = "../input/ventilator-pressure-prediction/train.csv"
TEST_PATH = "../input/ventilator-pressure-prediction/test.csv"
SAMPLE_PATH = "../input/ventilator-pressure-prediction/sample_submission.csv"

train_usecols = ["breath_id", "R", "C", "time_step", "u_in", "u_out", "pressure"]
train_dtypes = {
    "breath_id": "int32",
    "R": "int16",
    "C": "int16",
    "time_step": "float32",
    "u_in": "float32",
    "u_out": "int8",
    "pressure": "float32",
}
df_train = pd.read_csv(TRAIN_PATH, usecols=train_usecols, dtype=train_dtypes)

unique_pressures = df_train["pressure"].unique()
sorted_pressures = np.sort(unique_pressures)
total_pressures_len = len(sorted_pressures)


def find_nearest(prediction):
    insert_idx = np.searchsorted(sorted_pressures, prediction)
    if insert_idx == total_pressures_len:
        return sorted_pressures[-1]
    elif insert_idx == 0:
        return sorted_pressures[0]
    lower_val = sorted_pressures[insert_idx - 1]
    upper_val = sorted_pressures[insert_idx]
    return (
        lower_val
        if abs(lower_val - prediction) < abs(upper_val - prediction)
        else upper_val
    )


def snap_to_nearest_pressure_vec(pred_arr: np.ndarray) -> np.ndarray:
    pred = np.asarray(pred_arr, dtype=np.float64)
    idx = np.searchsorted(sorted_pressures, pred, side="left")
    idx1 = np.clip(idx, 0, total_pressures_len - 1)
    idx0 = np.clip(idx - 1, 0, total_pressures_len - 1)
    p0 = sorted_pressures[idx0]
    p1 = sorted_pressures[idx1]
    choose1 = np.abs(pred - p1) < np.abs(pred - p0)  # tie -> p0 (same as find_nearest)
    return np.where(choose1, p1, p0)


def set_seed(seed=2021):
    np.random.seed(seed)
    random_state = np.random.RandomState(seed)
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    return random_state


def wc(input_list):
    """
    Original intent: read one or two submissions and do a simple weighted combo.
    Bugfix: make parsing robust; if the filename does not contain a score token,
    just treat both weights equally (this preserves the blending idea without crashing).
    """
    l = []
    arrs = []
    for i in range(len(input_list)):
        fp = input_list[i]
        try:
            public_lb_score = int(fp.split("/")[-1].split(".")[1].split(" ")[0])
        except Exception:
            public_lb_score = 1
        l.append(public_lb_score)

        dfp = pd.read_csv(fp)
        if "pressure" not in dfp.columns:
            raise ValueError(f"File {fp} does not have 'pressure' column.")
        arrs.append(dfp["pressure"].to_numpy().ravel())

    l_sum = sum(l) if sum(l) != 0 else len(l)

    if len(arrs) == 1:
        return arrs[0]
    else:
        weight1 = (l[1] / l_sum) + 0.1
        weight2 = 1 - weight1
        return arrs[0] * weight1 + arrs[1] * weight2


def _safe_list_pred_files(dp):
    """
    Bugfix: in this environment the external dataset path may not exist.
    Only return readable .csv files; ignore directories/non-csv.
    """
    if dp is None:
        return []
    if not os.path.exists(dp):
        return []
    files = []
    for p in glob.iglob(f"{dp}/*"):
        if os.path.isfile(p) and p.lower().endswith(".csv"):
            files.append(p)
    files.sort()
    return files


def _load_pressure_vector(fp, expected_len):
    """
    Load a submission-like CSV and return pressure vector if shape matches.
    Skip invalid files rather than breaking the run.
    """
    try:
        dfp = pd.read_csv(fp, usecols=["pressure"])
        v = dfp["pressure"].to_numpy().ravel()
        if len(v) != expected_len:
            return None
        return v
    except Exception:
        return None


def _add_time_index_inplace_sorted(df):
    # Keep stable, deterministic order within breath
    df.sort_values(["breath_id", "time_step"], kind="mergesort", inplace=True)
    df["t_idx"] = df.groupby("breath_id", sort=False).cumcount().astype(np.int16)

    # Score-critical: enforce the known 80-step structure so train/test keys match perfectly
    # (prevents rare misalignment from any duplicated/float-order edge cases).
    df["t_idx"] = df["t_idx"].clip(0, 79).astype(np.int16)
    return df


def _quantize_to_int(series, scale=10.0, clip_min=-500000, clip_max=500000):
    """
    Score-critical: increase key match rate by quantizing to integer bins deterministically.
    Use int32 to avoid overflow for cumsum-based features (int16 overflow caused silent key noise).
    scale=10 => 0.1 resolution.
    """
    x = np.rint(series.to_numpy(dtype=np.float64, copy=False) * scale)
    x = np.clip(x, clip_min, clip_max)
    return x.astype(np.int32)


def _add_lag_features_fast_inplace_sorted(df, lags=(1, 2, 3, 5)):
    g = df.groupby("breath_id", sort=False)
    uin = g["u_in"]
    uout = g["u_out"]
    ts = g["time_step"]

    for k in lags:
        df[f"u_in_lag{k}"] = uin.shift(k)
        df[f"u_out_lag{k}"] = uout.shift(k)
        df[f"dt_lag{k}"] = ts.diff(k)

    df["u_in_diff1"] = uin.diff(1)
    df["u_in_diff2"] = uin.diff(2)
    df["u_in_cumsum"] = uin.cumsum()

    df.fillna(0.0, inplace=True)
    return df


def _nn_lookup_1d_by_group_vectorized(
    train_df, test_df, group_cols, xcol, ycol, outcol
):
    out = test_df.copy()
    out[outcol] = np.nan
    if len(train_df) == 0 or len(test_df) == 0:
        return out

    tr_keys = pd.MultiIndex.from_frame(train_df[group_cols])
    te_keys = pd.MultiIndex.from_frame(test_df[group_cols])
    all_keys = tr_keys.append(te_keys)
    key_codes, _ = pd.factorize(all_keys, sort=False)
    tr_code = key_codes[: len(train_df)].astype(np.int32, copy=False)
    te_code = key_codes[len(train_df) :].astype(np.int32, copy=False)

    x_tr = train_df[xcol].to_numpy()
    y_tr = train_df[ycol].to_numpy()
    x_te = test_df[xcol].to_numpy()

    order = np.lexsort((x_tr, tr_code))
    tr_code_s = tr_code[order]
    x_tr_s = x_tr[order]
    y_tr_s = y_tr[order]

    uniq_tr, start_idx, counts = np.unique(
        tr_code_s, return_index=True, return_counts=True
    )
    end_idx = start_idx + counts

    out_arr = np.full(len(test_df), np.nan, dtype=np.float32)

    for gcode, s, e in zip(uniq_tr, start_idx, end_idx):
        te_mask = te_code == gcode
        if not np.any(te_mask):
            continue
        xg = x_tr_s[s:e]
        yg = y_tr_s[s:e]
        xq = x_te[te_mask]

        idx = np.searchsorted(xg, xq, side="left")
        idx0 = np.clip(idx - 1, 0, len(xg) - 1)
        idx1 = np.clip(idx, 0, len(xg) - 1)

        d0 = np.abs(xq - xg[idx0])
        d1 = np.abs(xq - xg[idx1])
        choose1 = d1 < d0  # tie -> idx0

        pick = np.where(choose1, idx1, idx0)
        out_arr[te_mask] = yg[pick].astype(np.float32, copy=False)

    out[outcol] = out_arr
    return out


def _left_join_series_by_keys(df, ref_df, key_cols, value_col, out_col, agg="median"):
    if len(ref_df) == 0:
        df[out_col] = np.nan
        return df
    s = getattr(ref_df.groupby(key_cols, sort=False)[value_col], agg)()
    s = s.rename(out_col)
    df = df.join(s, on=key_cols)
    return df


def _fallback_pressure_lookup_submission(output_df, u_in_round=1):
    """
    Deterministic lookup with backoffs, then snap to nearest valid pressure.
    Inspiratory-only train lookups (u_out==0). Test u_out==1 rows set to 0.0.
    """

    test_usecols = ["id", "breath_id", "R", "C", "time_step", "u_out", "u_in"]
    test_dtypes = {
        "id": "int32",
        "breath_id": "int32",
        "R": "int16",
        "C": "int16",
        "time_step": "float32",
        "u_in": "float32",
        "u_out": "int8",
    }
    test = pd.read_csv(TEST_PATH, usecols=test_usecols, dtype=test_dtypes)

    train_insp = df_train.loc[df_train["u_out"].eq(0)].copy()

    _add_time_index_inplace_sorted(train_insp)
    _add_time_index_inplace_sorted(test)

    _add_lag_features_fast_inplace_sorted(train_insp, lags=(1, 2, 3, 5))
    _add_lag_features_fast_inplace_sorted(test, lags=(1, 2, 3, 5))

    q_cols = [
        "u_in",
        "u_in_lag1",
        "u_in_lag2",
        "u_in_lag3",
        "u_in_lag5",
        "u_in_diff1",
        "u_in_diff2",
        "u_in_cumsum",
    ]
    for c in q_cols:
        train_insp[c + "_q"] = _quantize_to_int(train_insp[c], scale=10.0)
        test[c + "_q"] = _quantize_to_int(test[c], scale=10.0)

    global_med = float(train_insp["pressure"].median())

    # Score-critical: add a slightly more specific early group so fewer rows fall back to coarse medians.
    grp00a = ["R", "C", "t_idx", "u_out", "u_in_lag1_q"]
    grp00 = [
        "R",
        "C",
        "t_idx",
        "u_out",
        "u_in_lag1_q",
        "u_in_lag2_q",
        "u_in_diff1_q",
        "u_in_diff2_q",
    ]
    grp0 = [
        "R",
        "C",
        "t_idx",
        "u_out",
        "u_in_lag1_q",
        "u_in_lag2_q",
        "u_in_lag3_q",
        "u_in_lag5_q",
        "u_in_diff1_q",
    ]
    grp1 = [
        "R",
        "C",
        "t_idx",
        "u_out",
        "u_in_lag1_q",
        "u_in_lag2_q",
        "u_in_diff1_q",
        "u_in_cumsum_q",
        "u_out_lag1",
        "u_out_lag2",
    ]
    grp2 = [
        "R",
        "C",
        "t_idx",
        "u_out",
        "u_in_lag1_q",
        "u_in_lag2_q",
        "u_in_diff1_q",
        "u_in_cumsum_q",
    ]

    m = test.copy()

    m = _nn_lookup_1d_by_group_vectorized(
        train_insp, m, grp00a, "u_in_q", "pressure", "pq00a"
    )
    m = _nn_lookup_1d_by_group_vectorized(
        train_insp, m, grp00, "u_in_q", "pressure", "pq00"
    )
    m = _nn_lookup_1d_by_group_vectorized(
        train_insp, m, grp0, "u_in_q", "pressure", "pq0"
    )
    m = _nn_lookup_1d_by_group_vectorized(
        train_insp, m, grp1, "u_in_q", "pressure", "pq1"
    )
    m = _nn_lookup_1d_by_group_vectorized(
        train_insp, m, grp2, "u_in_q", "pressure", "pq2"
    )

    key_q = ["R", "C", "t_idx", "u_out", "u_in_q"]
    key_qb = ["R", "C", "t_idx", "u_out", "u_in_q", "u_in_lag1_q", "u_in_lag2_q"]
    key_qc = ["R", "C", "t_idx", "u_out", "u_in_q", "u_in_diff1_q"]
    key4 = ["R", "C", "t_idx", "u_out"]
    key5 = ["R", "C", "t_idx"]

    m = _left_join_series_by_keys(m, train_insp, key_q, "pressure", "pq", agg="median")
    m = _left_join_series_by_keys(
        m, train_insp, key_qb, "pressure", "pqb", agg="median"
    )
    m = _left_join_series_by_keys(
        m, train_insp, key_qc, "pressure", "pqc", agg="median"
    )
    m = _left_join_series_by_keys(m, train_insp, key4, "pressure", "p4", agg="median")
    m = _left_join_series_by_keys(m, train_insp, key5, "pressure", "p5", agg="median")

    pred = m["pq00a"]
    pred = pred.fillna(m["pq00"])
    pred = pred.fillna(m["pq0"])
    pred = pred.fillna(m["pq1"])
    pred = pred.fillna(m["pq2"])
    pred = pred.fillna(m["pq"])
    pred = pred.fillna(m["pqb"])
    pred = pred.fillna(m["pqc"])
    pred = pred.fillna(m["p4"])
    pred = pred.fillna(m["p5"])
    pred = pred.fillna(global_med)

    output_df = output_df.copy()
    out_pred = pred.to_numpy(dtype=np.float64, copy=False)

    out_pred[m["u_out"].to_numpy() == 1] = 0.0

    output_df["pressure"] = snap_to_nearest_pressure_vec(out_pred)
    return output_df


def g(dp):
    """
    Blend if external prediction files exist; otherwise deterministic fallback lookup.
    """
    output = pd.read_csv(SAMPLE_PATH)
    expected_len = len(output)

    l = _safe_list_pred_files(dp)
    if len(l) == 0:
        output2 = _fallback_pressure_lookup_submission(output, u_in_round=1)
        output2.to_csv("submission.csv", index=False)
        return

    file_count = len(l)
    loop_time = 154
    splits = max(1, file_count // 2)

    flist_paths = []
    for i in range(splits):
        start = i * round(len(l) / splits)
        end = None if i == splits - 1 else (i + 1) * round(len(l) / splits)
        chunk = l[start:end]
        if len(chunk) > 0:
            flist_paths.append(chunk)

    flist = []
    for chunk in flist_paths:
        valid_chunk = []
        for fp in chunk:
            v = _load_pressure_vector(fp, expected_len)
            if v is not None:
                valid_chunk.append(fp)
        if len(valid_chunk) == 0:
            continue
        try:
            vec = wc(valid_chunk)
            if len(vec) == expected_len:
                flist.append(vec.astype(np.float64, copy=False))
        except Exception:
            continue

    if len(flist) == 0:
        output2 = _fallback_pressure_lookup_submission(output, u_in_round=1)
        output2.to_csv("submission.csv", index=False)
        return

    n_models = len(flist)
    preds_mat = np.empty((loop_time, expected_len), dtype=np.float64)

    for loop_idx in range(loop_time):
        set_seed(loop_idx)
        w = np.fromiter(
            (rd() for _ in range(n_models)), dtype=np.float64, count=n_models
        )
        ws = w.sum()
        if ws == 0:
            w[:] = 1.0 / n_models
        else:
            w /= ws
        w.sort()
        w = w[::-1]

        temp = np.zeros(expected_len, dtype=np.float64)
        for j in range(n_models):
            temp += flist[j] * w[j]
        preds_mat[loop_idx] = temp

    output_pred = np.median(preds_mat, axis=0)
    output["pressure"] = snap_to_nearest_pressure_vec(output_pred)

    output.to_csv(f"rwb_{loop_time}_loops.csv", index=False)
    output.to_csv("submission.csv", index=False)


def blend(a, b):
    a = pd.read_csv(a)
    b = pd.read_csv(b)
    a.pressure = a.pressure * 0.7 + b.pressure * 0.3
    a["pressure"] = snap_to_nearest_pressure_vec(
        a["pressure"].to_numpy(dtype=np.float64, copy=False)
    )
    a.to_csv("blend.csv", index=False)
    return a




In [3]:
g("../input/gb-data-blending-recover")